# Portfolio returns and multi-period attribution

This notebook covers holdings-based performance measurement and attribution exposed by `finstack_quant.portfolio`:

- **Time-weighted return (TWRR)** via Modified Dietz per sub-period, then geometric linking.
- **Money-weighted return (MWRR)** via XIRR on dated cashflows.
- **Brinson-Fachler attribution** -- splitting active return into allocation, selection, and interaction by sector for a single period.
- **Carino linking** -- combining single-period Brinson effects into multi-period contributions that reconcile to the geometric active return.

All of these functions are JSON-in / JSON-out (or scalar), so specs and results log and replay unchanged.

## Imports

In [1]:
import json

import pandas as pd

from finstack_quant.portfolio import (
    twrr_modified_dietz,
    twrr_linked,
    mwr_xirr,
    brinson_fachler,
    carino_link,
)

## Time-weighted return (Modified Dietz per sub-period)

Each sub-period is a beginning market value (BMV), an ending market value (EMV), and external cashflows. For TWRR the **sign convention is portfolio-centric**: a contribution into the portfolio is **positive**, a withdrawal is **negative**. Each flow carries `fraction_of_period_remaining` -- the Dietz time weight in `[0, 1]` (a start-of-period flow is `1.0`, an end-of-period flow is `0.0`).

The Modified Dietz return is `(EMV - BMV - sum(CF)) / (BMV + sum(w * CF))`.

In [2]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/portfolio_returns_and_attribution.json").read_text())

sub_periods = _NOTEBOOK_DATA['sub_periods']

sub_returns = [twrr_modified_dietz(json.dumps(p)) for p in sub_periods]
pd.DataFrame({"sub_period": ["Q1", "Q2"], "modified_dietz_return": sub_returns})

,sub_period,modified_dietz_return
0,Q1,-0.047170
1,Q2,0.014563


### Linking sub-period returns

`twrr_linked` compounds the sub-period returns geometrically (`prod(1 + r) - 1`). `horizon_years` is the elapsed time as a fraction of a 365-day year; annualisation only applies once the horizon reaches a full year.

In [3]:
half_year = json.loads(twrr_linked(json.dumps(sub_returns), 0.5))
print("Two quarters (horizon 0.5y):")
print(f"  cumulative = {half_year['cumulative']:.4%}   annualised = {half_year['annualised']:.4%}   periods = {half_year['num_periods']}")

# Over a longer horizon, annualisation differs from the cumulative figure
six_quarters = [0.02, 0.015, -0.01, 0.03, 0.012, 0.018]
linked = json.loads(twrr_linked(json.dumps(six_quarters), 1.5))
print("\nSix quarters (horizon 1.5y):")
print(f"  cumulative = {linked['cumulative']:.4%}   annualised = {linked['annualised']:.4%}   periods = {linked['num_periods']}")

Two quarters (horizon 0.5y):
  cumulative = -3.3294%   annualised = -3.3294%   periods = 2

Six quarters (horizon 1.5y):
  cumulative = 8.7594%   annualised = 5.7575%   periods = 6


## Money-weighted return (XIRR)

MWRR is the internal rate of return that prices the dated external cashflows to zero. Here the **sign convention is investor-centric and the opposite of TWRR**: money paid *into* the portfolio is **negative**, terminal value and distributions received are **positive**.

In [4]:
cashflows = [
    {"date": "2025-01-01", "amount": -10_000_000.0},  # initial investment
    {"date": "2025-04-01", "amount": -1_000_000.0},   # contribution
    {"date": "2025-10-01", "amount": 500_000.0},      # distribution
    {"date": "2026-01-01", "amount": 11_200_000.0},   # terminal value
]

irr = mwr_xirr(json.dumps(cashflows))
print(f"Money-weighted return (annualised XIRR): {irr:.4%}")

Money-weighted return (annualised XIRR): 6.5886%


TWRR strips out the effect of cashflow timing (it measures the manager's decisions), while MWRR reflects the investor's actual dollar-weighted experience. The two diverge whenever flows are large and poorly / well timed relative to performance.

## Brinson-Fachler attribution (single period)

Given portfolio and benchmark weights and returns per sector, Brinson-Fachler decomposes the active return into three effects per sector:

- **Allocation** $= (w_p - w_b)(r_b^{i} - r_b)$
- **Selection** $= w_b (r_p^{i} - r_b^{i})$
- **Interaction** $= (w_p - w_b)(r_p^{i} - r_b^{i})$

Weights must sum to 1.0 on each side.

In [5]:
def sector(name, wp, wb, rp, rb):
    return {
        "sector": name,
        "portfolio_weight": wp,
        "benchmark_weight": wb,
        "portfolio_return": rp,
        "benchmark_return": rb,
    }

period_1 = [
    sector("Tech", 0.45, 0.40, 0.085, 0.060),
    sector("Energy", 0.30, 0.35, 0.010, 0.030),
    sector("Health", 0.25, 0.25, 0.040, 0.035),
]

bf = json.loads(brinson_fachler(json.dumps(period_1)))
pd.DataFrame(bf["sectors"])

,sector,allocation,selection,interaction,total
0,Tech,0.000837,0.01000,0.00125,0.012088
1,Energy,0.000662,-0.00700,0.00100,-0.005337
2,Health,-0.000000,0.00125,0.00000,0.001250


In [6]:
summary = {
    "total_allocation": bf["total_allocation"],
    "total_selection": bf["total_selection"],
    "total_interaction": bf["total_interaction"],
    "total_excess_return": bf["total_excess_return"],
    "portfolio_return": bf["portfolio_return"],
    "benchmark_return": bf["benchmark_return"],
}
recon = bf["total_allocation"] + bf["total_selection"] + bf["total_interaction"]
print(f"allocation + selection + interaction = {recon:.4%}")
print(f"total excess return                  = {bf['total_excess_return']:.4%}")
pd.Series(summary)

allocation + selection + interaction = 0.8000%
total excess return                  = 0.8000%


total_allocation       0.00150
total_selection        0.00425
total_interaction      0.00225
total_excess_return    0.00800
portfolio_return       0.05125
benchmark_return       0.04325
dtype: float64

## Multi-period attribution with Carino linking

Single-period effects do not add up arithmetically across periods (returns compound). Carino linking applies a smoothing coefficient per period so that the linked allocation / selection / interaction effects sum exactly to the **geometric** active return over the full horizon.

In [7]:
period_2 = [
    sector("Tech", 0.50, 0.40, 0.020, 0.030),
    sector("Energy", 0.25, 0.35, -0.010, 0.000),
    sector("Health", 0.25, 0.25, 0.015, 0.010),
]
period_3 = [
    sector("Tech", 0.40, 0.40, 0.030, 0.025),
    sector("Energy", 0.35, 0.35, 0.020, 0.015),
    sector("Health", 0.25, 0.25, 0.010, 0.012),
]

linked_attr = json.loads(carino_link(json.dumps([period_1, period_2, period_3])))
pd.DataFrame(linked_attr["linked_sectors"])

,sector,allocation,selection,interaction,total
0,Tech,0.002521,0.008179,0.000223,0.010923
1,Energy,0.002233,-0.009113,0.002101,-0.004779
2,Health,0.000000,0.002096,0.000000,0.002096


In [8]:
geometric_active = linked_attr["portfolio_return_compounded"] - linked_attr["benchmark_return_compounded"]
recon = (
    linked_attr["linked_allocation"]
    + linked_attr["linked_selection"]
    + linked_attr["linked_interaction"]
)
print(f"compounded portfolio return : {linked_attr['portfolio_return_compounded']:.4%}")
print(f"compounded benchmark return : {linked_attr['benchmark_return_compounded']:.4%}")
print(f"geometric active return     : {geometric_active:.4%}")
print(f"sum of linked effects       : {recon:.4%}")
print(f"reconciles: {abs(recon - geometric_active) < 1e-9}")

compounded portfolio return : 8.5933%
compounded benchmark return : 7.7693%
geometric active return     : 0.8240%
sum of linked effects       : 0.8240%
reconciles: True


## Takeaways

- `twrr_modified_dietz` computes a single sub-period TWRR (contributions positive); `twrr_linked` compounds sub-period returns and annualises once the horizon reaches a year.
- `mwr_xirr` computes the money-weighted (dollar-weighted) return; its cashflow signs are the opposite of the TWRR convention (money in is negative).
- `brinson_fachler` splits a single period's active return into allocation, selection, and interaction by sector, reconciling exactly to the total excess return.
- `carino_link` chains single-period Brinson results so the linked effects reconcile to the geometric active return across the whole horizon.